# 08 — Mixture of Experts (MoE)
### Starter Notebook

**Learning objectives**
- Understand why MoE lets you scale parameters without scaling compute
- Implement a simple top-k router
- Understand load-balancing loss and why it's needed
- Use the library's MoE layer in a transformer block

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from typing import Tuple

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Part A — The MoE Idea

In a dense transformer, **every** token passes through **every** parameter at every layer.

In an MoE transformer, the FFN layer is replaced by $N$ **expert** networks. A **router** picks the top-$k$ experts for each token. Only those $k$ experts activate — so **parameters scale**, but **compute stays fixed**.

Used in: **Gemini**, **GPT-4** (rumoured), **Mixtral**, **Switch Transformer**.

## Part B — Implement a Simple Router

### Exercise B1

In [ ]:
class SimpleTopKRouter(nn.Module):
    """
    Routes each token to the top-k experts.

    Args:
        d_model     : model dimension
        num_experts : total experts available
        top_k       : how many to activate per token
    """

    def __init__(self, d_model: int, num_experts: int, top_k: int = 2):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts
        # TODO: linear gate: d_model → num_experts (no bias)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            weights : (batch, seq_len, top_k)  — softmax-normalised weights for chosen experts
            indices : (batch, seq_len, top_k)  — which experts were chosen
            logits  : (batch, seq_len, num_experts) — raw logits for aux loss
        """
        # TODO:
        # 1. gate → logits (batch, seq, num_experts)
        # 2. top-k selection: use torch.topk
        # 3. normalise selected weights with softmax
        pass


router = SimpleTopKRouter(d_model=64, num_experts=8, top_k=2)
x_test = torch.randn(2, 10, 64)
result = router(x_test)
if result is not None:
    weights, indices, logits = result
    print(f'Weights shape : {weights.shape}   # (2, 10, 2)')
    print(f'Indices shape : {indices.shape}')
    print(f'Logits shape  : {logits.shape}')
    print(f'Weights sum   : {weights.sum(-1).mean():.4f}  # should be 1.0')

## Part C — The Routing Collapse Problem

Without regularisation, the router tends to always pick the same 1-2 experts — others get no gradient and atrophy. We need a **load-balancing loss**.

### Exercise C1 — Implement auxiliary load-balancing loss

In [ ]:
def load_balancing_loss(
    router_logits: torch.Tensor,   # (batch, seq, num_experts)
    expert_indices: torch.Tensor,  # (batch, seq, top_k)
    num_experts: int,
) -> torch.Tensor:
    """
    Auxiliary loss to encourage balanced expert usage.

    From 'Switch Transformers' (Fedus et al., 2022):
        L_aux = num_experts * sum_i(f_i * P_i)

    where:
        f_i = fraction of tokens routed to expert i
        P_i = mean router probability for expert i

    A perfectly balanced router has L_aux = 1.0.
    """
    batch, seq_len, top_k = expert_indices.shape
    total_tokens = batch * seq_len

    # TODO:
    # 1. Count how many tokens were routed to each expert → f_i
    # 2. Compute mean softmax prob for each expert → P_i
    # 3. Return num_experts * dot(f, P)
    pass


# Test: simulate balanced vs collapsed routing
router_test = SimpleTopKRouter(d_model=64, num_experts=8, top_k=2)
x_test = torch.randn(4, 20, 64)
result = router_test(x_test)
if result is not None:
    weights, indices, logits = result

    # Count how often each expert is selected
    counts = torch.zeros(8)
    for e in range(8):
        counts[e] = (indices == e).float().sum()

    plt.figure(figsize=(7, 3))
    plt.bar(range(8), counts.detach().numpy())
    plt.xlabel('Expert ID'); plt.ylabel('Tokens assigned')
    plt.title('Expert Load Distribution (random weights — should be roughly uniform)')
    plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## Part D — Full MoE Layer

In [ ]:
from src.models.moe import MoELayer, MoEBlock

moe = MoELayer(d_model=128, num_experts=8, top_k=2)
x = torch.randn(2, 16, 128)
out, aux_loss = moe(x)
print(f'MoELayer output : {out.shape}')
print(f'Auxiliary loss  : {aux_loss.item():.4f}  # ~1.0 = balanced')

# Full MoE transformer block
moe_block = MoEBlock(d_model=128, n_heads=4, num_experts=8, top_k=2)
out2, aux2 = moe_block(x)
print(f'MoEBlock output : {out2.shape}')

## Part E — Parameter vs Compute Trade-off

In [ ]:
from src.models.feedforward import FeedForward

d_model, d_ff, num_experts, top_k = 512, 2048, 8, 2

# Dense FFN
dense = FeedForward(d_model, d_ff)
dense_params = sum(p.numel() for p in dense.parameters())

# MoE FFN: same per-expert size, but N experts
# Only top_k activate at inference
per_expert_params = dense_params
moe_total_params  = per_expert_params * num_experts
moe_active_params = per_expert_params * top_k   # compute at inference

print(f'Dense FFN:')
print(f'  Parameters : {dense_params:,}')
print(f'  Active     : {dense_params:,}  (100%)')
print()
print(f'MoE FFN ({num_experts} experts, top-{top_k}):')
print(f'  Total params   : {moe_total_params:,}   ({moe_total_params/dense_params:.0f}x more)')
print(f'  Active params  : {moe_active_params:,}   (same compute as dense with {top_k} experts)')
print(f'  Param efficiency: {top_k}/{num_experts} = {top_k/num_experts:.1%} of experts active per token')

## Summary

- **MoE** decouples model size from compute cost — you can have 8x more parameters at the same FLOPs.
- The **router** learns which expert is best for each token — different experts may specialise in different linguistic phenomena.
- **Load balancing loss** prevents expert collapse — all experts must contribute.
- Mixtral-8x7B = 8 experts, top-2 → 47B total params, 13B active per token.

**Next:** `09_multimodal_fusion_starter.ipynb` — extending the transformer to images.